## Environment Setup & Dependencies

In [1]:
import os
import cv2
import librosa
import numpy as np
import pandas as pd

from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import tensorflow as tf

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Conv2D,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Concatenate
)

from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

## 2D Representation: Mel-Spectrogram Extraction
Convert raw 1D audio waveforms into 2D visual representations. The Mel-spectrograms are converted to a decibel scale, min-max normalized per sample, and resized to fixed 128x128 dimensions for CNN compatibility.

In [2]:
def get_mel_spectrogram(y, sr):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # normalize per sample
    mel_db = (mel_db - np.min(mel_db)) / (np.max(mel_db) - np.min(mel_db) + 1e-6)

    # Resize to fixed size
    mel_db = cv2.resize(mel_db, (128, 128))

    return mel_db

In [3]:
def extract_numerical_features(y, sr):

    feature_dict = {}

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=16)
    mfcc_feat = np.concatenate([
        np.mean(mfcc, axis=1),
        np.std(mfcc, axis=1)
    ])
    feature_dict["mfcc"] = mfcc_feat

    # SPECTRAL CENTROID
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_feat = np.array([
        np.mean(centroid),
        np.std(centroid)
    ])
    feature_dict["centroid"] = centroid_feat

    # RMS
    rms = librosa.feature.rms(y=y)
    rms_feat = np.array([
        np.mean(rms),
        np.std(rms)
    ])
    feature_dict["rms"] = rms_feat

    # ZCR
    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_feat = np.array([
        np.mean(zcr),
        np.std(zcr)
    ])
    feature_dict["zcr"] = zcr_feat

    # PITCH
    f0 = librosa.yin(y, fmin=50, fmax=1000)
    f0 = f0[~np.isnan(f0)]

    if len(f0) > 0:
        pitch_feat = np.array([
            np.mean(f0),
            np.std(f0)
        ])
    else:
        pitch_feat = np.array([0, 0])

    feature_dict["pitch"] = pitch_feat

    return feature_dict

## Dual-Modal Data Ingestion
Iterate through the dataset to simultaneously extract both the 2D Mel-spectrograms and the 1D numerical feature arrays. This prepares the aligned inputs required for the multi-input fusion model.

In [5]:
X_spec = []
mfcc_list = []
centroid_list = []
rms_list = []
zcr_list = []
pitch_list = []

y_labels = []

def process_folder(folder_path, label):
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)

        try:
            # LOAD AUDIO
            y_audio, sr = librosa.load(file_path, sr=16000)

            # MEL-SPECTROGRAM
            mel_spec = get_mel_spectrogram(y_audio, sr)
            X_spec.append(mel_spec)

            # NUMERICAL FEATURES
            feats = extract_numerical_features(y_audio, sr)

            mfcc_list.append(feats["mfcc"])
            centroid_list.append(feats["centroid"])
            rms_list.append(feats["rms"])
            zcr_list.append(feats["zcr"])
            pitch_list.append(feats["pitch"])
            y_labels.append(label)

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

process_folder("dataset/normal", 0)
process_folder("dataset/sepsis", 1)

## Convert to NumPy Arrays

In [6]:
X_spec = np.array(X_spec)
mfcc_list = np.array(mfcc_list)
centroid_list = np.array(centroid_list)
rms_list = np.array(rms_list)
zcr_list = np.array(zcr_list)
pitch_list = np.array(pitch_list)
y = np.array(y_labels)

# Add CNN Channel Dimension
X_spec = X_spec[..., np.newaxis]

print(X_spec.shape)
print(mfcc_list.shape)
print(centroid_list.shape)
print(rms_list.shape)
print(zcr_list.shape)
print(pitch_list.shape)
print(y.shape)

(1817, 128, 128, 1)
(1817, 32)
(1817, 2)
(1817, 2)
(1817, 2)
(1817, 2)
(1817,)


In [7]:
top_feature_combos = [
    ("mfcc", "rms", "zcr"),
    ("mfcc", "rms", "pitch"),
    ("mfcc", "pitch"),
    ("mfcc", "centroid", "rms", "zcr"),
    ("mfcc", "centroid", "pitch")
]

feature_dict = {
    "mfcc": mfcc_list,
    "centroid": centroid_list,
    "rms": rms_list,
    "zcr": zcr_list,
    "pitch": pitch_list
}

## Multi-Modal Fusion Architecture (CNN + MLP)
Define a dual-branch neural network:
* **CNN Branch**: Processes the 128x128x1 Mel-spectrograms through sequential Conv2D and MaxPooling layers, flattened via Global Average Pooling.
* **Feature Branch**: Processes the 1D numerical inputs through Dense layers.
* **Fusion Head**: Concatenates both branches into a unified dense network terminating in a sigmoid classifier.

In [8]:
def build_fusion_model(feature_dim):

    # =================================
    # CNN BRANCH
    # =================================
    input_img = Input(shape=(128,128,1))

    x = Conv2D(32, (3,3), activation='relu', padding='same')(input_img)
    x = MaxPooling2D(2,2)(x)
    x = Dropout(0.1)(x)

    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D(2,2)(x)
    x = Dropout(0.2)(x)

    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D(2,2)(x)
    x = Dropout(0.3)(x)

    x = GlobalAveragePooling2D()(x)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)

    # =================================
    # FEATURE BRANCH
    # =================================
    input_feat = Input(shape=(feature_dim,))

    f = Dense(128, activation='relu')(input_feat)
    f = Dropout(0.3)(f)

    f = Dense(64, activation='relu')(f)
    f = Dropout(0.2)(f)

    # =================================
    # FUSION
    # =================================
    combined = Concatenate()([x, f])

    z = Dense(128, activation='relu')(combined)
    z = Dropout(0.3)(z)

    z = Dense(64, activation='relu')(z)
    z = Dropout(0.2)(z)

    output = Dense(1, activation='sigmoid')(z)

    # =================================
    # MODEL
    # =================================
    model = Model(inputs=[input_img, input_feat], outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=0.0003),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

## Top Combinations Multi-Seed Evaluation
Evaluate the fusion architecture using the top 5 numerical feature combinations identified in the baseline ablation study. Each configuration is validated across three random splits to determine the architecture's stability and aggregated recall performance.

In [11]:
random_states = [42, 357, 999]
fusion_results = []

for combo in top_feature_combos:

    print("\n===================================")
    print("Training Fusion Model on:", combo)

    # CONCAT FEATURES
    X_feat = np.concatenate(
        [feature_dict[f] for f in combo],
        axis=1
    )
    print("Feature Shape:", X_feat.shape)

    # METRIC STORAGE
    acc_list = []
    precision_list = []
    recall_list = []
    f1_list = []
    auc_list = []

    # 3 RANDOM SPLITS
    for rs in random_states:
        print(f"\n--- Random State: {rs} ---")

        (
            X_spec_train,
            X_spec_test,

            X_feat_train,
            X_feat_test,

            y_train,
            y_test

        ) = train_test_split(

            X_spec,
            X_feat,
            y,

            test_size=0.2,
            random_state=rs,
            stratify=y
        )

        # STANDARDIZATION
        scaler = StandardScaler()
        X_feat_train = scaler.fit_transform(X_feat_train)
        X_feat_test = scaler.transform(X_feat_test)

        
        # BUILD MODEL
        model = build_fusion_model(feature_dim=X_feat_train.shape[1])

        # TRAIN
        model.fit(
            [X_spec_train, X_feat_train],
            y_train,

            validation_data=(
                [X_spec_test, X_feat_test],
                y_test
            ),

            epochs=10,
            batch_size=32,
            verbose=0
        )

        # PREDICTIONS
        threshold = 0.4
        y_prob = model.predict(
            [X_spec_test, X_feat_test],
            verbose=0
        )
        y_pred = (y_prob > threshold).astype(int).reshape(-1)

        # METRICS
        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_prob.ravel())

        # SAVE CURRENT SPLIT METRICS
        acc_list.append(acc)
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        auc_list.append(auc)

        print("Accuracy :", round(acc, 4))
        print("Precision:", round(precision, 4))
        print("Recall   :", round(recall, 4))
        print("F1 Score :", round(f1, 4))
        print("ROC AUC  :", round(auc, 4))

        K.clear_session()

    # AVERAGE RESULTS OF 3 SPLITS
    avg_acc = np.mean(acc_list)
    avg_precision = np.mean(precision_list)
    avg_recall = np.mean(recall_list)
    avg_f1 = np.mean(f1_list)
    avg_auc = np.mean(auc_list)

    std_acc = np.std(acc_list)
    std_precision = np.std(precision_list)
    std_recall = np.std(recall_list)
    std_f1 = np.std(f1_list)
    std_auc = np.std(auc_list)

    # SAVE FINAL RESULTS
    fusion_results.append({
        "Features": " + ".join(combo),
        "Accuracy": avg_acc,
        "Precision": avg_precision,
        "Recall": avg_recall,
        "F1": avg_f1,
        "ROC_AUC": avg_auc
    })

    # PRINT FINAL AVG RESULTS
    print("\n===== AVERAGE RESULTS =====")

    print("Avg Accuracy :", round(avg_acc, 4))
    print("Avg Precision:", round(avg_precision, 4))
    print("Avg Recall   :", round(avg_recall, 4))
    print("Avg F1 Score :", round(avg_f1, 4))
    print("Avg ROC AUC  :", round(avg_auc, 4))


Training Fusion Model on: ('mfcc', 'rms', 'zcr')
Feature Shape: (1817, 36)

--- Random State: 42 ---
Accuracy : 0.9643
Precision: 0.9647
Recall   : 0.9591
F1 Score : 0.9619
ROC AUC  : 0.994

--- Random State: 357 ---
Accuracy : 0.9725
Precision: 0.9497
Recall   : 0.9942
F1 Score : 0.9714
ROC AUC  : 0.9943

--- Random State: 999 ---
Accuracy : 0.9533
Precision: 0.9375
Recall   : 0.9649
F1 Score : 0.951
ROC AUC  : 0.9908

===== AVERAGE RESULTS =====
Avg Accuracy : 0.9634
Avg Precision: 0.9506
Avg Recall   : 0.9727
Avg F1 Score : 0.9614
Avg ROC AUC  : 0.993

Training Fusion Model on: ('mfcc', 'rms', 'pitch')
Feature Shape: (1817, 36)

--- Random State: 42 ---
Accuracy : 0.967
Precision: 0.9543
Recall   : 0.9766
F1 Score : 0.9653
ROC AUC  : 0.9962

--- Random State: 357 ---
Accuracy : 0.956
Precision: 0.9641
Recall   : 0.9415
F1 Score : 0.9527
ROC AUC  : 0.9939

--- Random State: 999 ---
Accuracy : 0.9533
Precision: 0.9477
Recall   : 0.9532
F1 Score : 0.9504
ROC AUC  : 0.9882

===== AVERA

## RESULTS DATAFRAME

In [12]:
fusion_results_df = pd.DataFrame(fusion_results)
fusion_results_df = fusion_results_df.sort_values(
    by="Recall",
    ascending=False
)
fusion_results_df

,Features,Accuracy,Precision,Recall,F1,ROC_AUC
2,mfcc + pitch,0.965201,0.950763,0.976608,0.963421,0.992031
0,mfcc + rms + zcr,0.963370,0.950642,0.972710,0.961438,0.993021
3,mfcc + centroid + rms + zcr,0.961538,0.953945,0.964912,0.959328,0.994445
4,mfcc + centroid + pitch,0.961538,0.954204,0.964912,0.959297,0.994667
1,mfcc + rms + pitch,0.958791,0.955344,0.957115,0.956139,0.992758


## Save CSV

In [14]:
fusion_results_df.to_csv(
    "fusion_results.csv",
    index=False
)

print("\nResults saved successfully!")


Results saved successfully!


## 5. Production Artifact Generation
Train the final fusion models for the top 5 configurations using a fixed seed. Export the serialized `.keras` models and their corresponding `.joblib` StandardScalers into the `artifacts/` directory for seamless integration with the FastAPI backend inference pipeline.

In [11]:
import os
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Create the artifacts directory
os.makedirs("artifacts", exist_ok=True)

# 2. Define the top 5 combinations
top_5_combos = [
    ("mfcc", "centroid", "pitch"),
    ("mfcc", "rms", "pitch"),
    ("mfcc", "pitch"),
    ("mfcc", "centroid", "rms", "zcr"),
    ("mfcc", "rms", "zcr")
]

# 3. Train and export loop
for combo in top_5_combos:
    pipeline_name = "_".join(combo)
    print(f"Training and exporting artifacts for: {pipeline_name}...")
    
    # Concatenate features
    X_feat_combo = np.concatenate([feature_dict[f] for f in combo], axis=1)
    
    # Train-Test Split (We use just ONE random state: 42 for the final production model)
    X_spec_train, X_spec_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(
        X_spec, X_feat_combo, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # STANDARDIZATION
    scaler = StandardScaler()
    X_feat_train = scaler.fit_transform(X_feat_train)
    X_feat_test = scaler.transform(X_feat_test)
    
    # BUILD MODEL (Using your exact function)
    model = build_fusion_model(feature_dim=X_feat_train.shape[1])
    
    # TRAIN (Using your exact parameters)
    model.fit(
        [X_spec_train, X_feat_train],
        y_train,
        validation_data=(
            [X_spec_test, X_feat_test],
            y_test
        ),
        epochs=10,
        batch_size=32,
        verbose=0
    )
    
    # EXPORT THE FILES
    model_filepath = f"artifacts/fusion_model_{pipeline_name}.keras"
    scaler_filepath = f"artifacts/feature_scaler_{pipeline_name}.joblib"
    
    model.save(model_filepath)
    joblib.dump(scaler, scaler_filepath)
    print(f" -> Saved Model: {model_filepath}")
    print(f" -> Saved Scaler: {scaler_filepath}")

print("\nSUCCESS: All 10 artifacts are ready for the backend!")

Training and exporting artifacts for: mfcc_centroid_pitch...
 -> Saved Model: artifacts/fusion_model_mfcc_centroid_pitch.keras
 -> Saved Scaler: artifacts/feature_scaler_mfcc_centroid_pitch.joblib
Training and exporting artifacts for: mfcc_rms_pitch...
 -> Saved Model: artifacts/fusion_model_mfcc_rms_pitch.keras
 -> Saved Scaler: artifacts/feature_scaler_mfcc_rms_pitch.joblib
Training and exporting artifacts for: mfcc_pitch...
 -> Saved Model: artifacts/fusion_model_mfcc_pitch.keras
 -> Saved Scaler: artifacts/feature_scaler_mfcc_pitch.joblib
Training and exporting artifacts for: mfcc_centroid_rms_zcr...
 -> Saved Model: artifacts/fusion_model_mfcc_centroid_rms_zcr.keras
 -> Saved Scaler: artifacts/feature_scaler_mfcc_centroid_rms_zcr.joblib
Training and exporting artifacts for: mfcc_rms_zcr...
 -> Saved Model: artifacts/fusion_model_mfcc_rms_zcr.keras
 -> Saved Scaler: artifacts/feature_scaler_mfcc_rms_zcr.joblib

SUCCESS: All 10 artifacts are ready for the backend!
